# Pipeline walkthrough

Runs the real pipeline against the local Postgres from `docker-compose.yml` and checks the two properties the project is built around: **incremental loading** (each run fetches only the dates after the last loaded one) and **idempotency** (loading the same data twice doesn't create duplicates).

Note: section 3 deletes the last week of rows from `exchange_rates` on purpose and lets the flow load them back.

In [1]:
import sys
sys.path.append("..")

from datetime import date, timedelta

from src.db import get_connection
from src.extract import fetch_rates_for_range
from src.flow import fx_rates_flow
from src.load import get_last_loaded_date, upsert_rates
from src.transform import parse_rates_response

In [2]:
def table_summary(conn):
    with conn.cursor() as cur:
        cur.execute("""
            select count(*), count(distinct (effective_date, currency_code)),
                   min(effective_date), max(effective_date)
            from exchange_rates
        """)
        rows, unique_keys, first_date, last_date = cur.fetchone()
    print(f"rows: {rows}, unique keys: {unique_keys}, dates: {first_date}..{last_date}")


conn = get_connection()

## 1. Current state of the table

In [3]:
table_summary(conn)

rows: 22839, unique keys: 22839, dates: 2024-01-02..2026-10-07


`rows` equal to `unique keys` means there are no duplicate `(effective_date, currency_code)` pairs. The primary key wouldn't allow them anyway, but it's a quick sanity check to repeat after every step below.

## 2. Run the flow

Same as `python -m src.flow`. On an empty table this backfills everything from `DEFAULT_START` (2024-01-01). If the table is already up to date, the flow should stop without calling the API.

In [4]:
fx_rates_flow()

15:39:23.988 | INFO    | prefect - Starting temporary server on http://127.0.0.1:8828
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.

15:39:30.949 | INFO    | Flow run 'amusing-copperhead' - Beginning flow run 'amusing-copperhead' for flow 'fx-rates-etl'

15:39:30.979 | INFO    | Task run 'get_last_loaded_date_task-171' - Finished in state Completed()

15:39:30.981 | INFO    | Flow run 'amusing-copperhead' - nothing to load, last loaded date is 2026-10-07

15:39:31.970 | INFO    | Flow run 'amusing-copperhead' - Finished in state Completed()

In [5]:
table_summary(conn)

rows: 22839, unique keys: 22839, dates: 2024-01-02..2026-10-07


The table was already loaded up to today, so the flow only read the watermark and stopped - no API call, no new rows.

## 3. Incremental load

Simulate a pipeline that hasn't run for a week: delete the last 7 days, then run the flow again. The watermark (`MAX(effective_date)`) moves back, so the flow should fetch exactly the missing window and nothing else.

In [6]:
last_date = get_last_loaded_date(conn)
cutoff = last_date - timedelta(days=7)

with conn.cursor() as cur:
    cur.execute("delete from exchange_rates where effective_date > %s", (cutoff,))
    deleted = cur.rowcount
conn.commit()

print(f"deleted {deleted} rows after {cutoff}")
table_summary(conn)

deleted 160 rows after 2026-09-30
rows: 22679, unique keys: 22679, dates: 2024-01-02..2026-09-30


In [7]:
fx_rates_flow()

15:39:32.178 | INFO    | Flow run 'splendid-guan' - Beginning flow run 'splendid-guan' for flow 'fx-rates-etl'

15:39:32.194 | INFO    | Task run 'get_last_loaded_date_task-50c' - Finished in state Completed()

15:39:32.342 | INFO    | Task run 'extract_task-380' - Finished in state Completed()

15:39:32.357 | INFO    | Task run 'transform_task-7a9' - Finished in state Completed()

15:39:32.369 | INFO    | Task run 'validate_task-601' - Finished in state Completed()

15:39:32.392 | INFO    | Task run 'load_task-6b9' - Finished in state Completed()

15:39:32.395 | INFO    | Flow run 'splendid-guan' - loaded 160 rows for 2026-10-01..2026-10-07

15:39:33.183 | INFO    | Flow run 'splendid-guan' - Finished in state Completed()

In [8]:
table_summary(conn)

rows: 22839, unique keys: 22839, dates: 2024-01-02..2026-10-07


160 rows deleted, 160 rows loaded back, and the window in the flow log (2026-10-01..2026-10-07) starts exactly one day after the new watermark. That's 5 business days x 32 currencies - NBP doesn't publish on weekends, so 3-4 October simply aren't in the response.

## 4. Idempotency

The watermark means the flow itself never re-fetches dates it already has. But a batch can still be loaded twice, e.g. a manual backfill of an old range, or a retry after a crash. Here January 2024 (already in the table) is fetched and upserted twice in a row.

In [9]:
raw_json = fetch_rates_for_range(date(2024, 1, 1), date(2024, 1, 31))
df = parse_rates_response(raw_json)
print(f"batch: {len(df)} rows")

table_summary(conn)
upsert_rates(conn, df)
upsert_rates(conn, df)
table_summary(conn)

batch: 726 rows


rows: 22839, unique keys: 22839, dates: 2024-01-02..2026-10-07
rows: 22839, unique keys: 22839, dates: 2024-01-02..2026-10-07


726 rows upserted twice and the row count didn't move. `INSERT ... ON CONFLICT DO UPDATE` turns a repeated load into an update of existing rows instead of an insert, so re-running any part of the pipeline is safe.

In [10]:
conn.close()